In [ ]:
import os
import json
import openai
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from openai import OpenAI
from langsmith import Client

In [29]:
load_dotenv(
    "/Users/benny/AI-Engineer/.env",
    override=True,
)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [10]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

In [11]:
print(qdrant_client.get_collections())

collections=[CollectionDescription(name='amazon_books_collection_sampled')]


### Download all data from Qdrant

In [16]:
all_points = qdrant_client.scroll(
    collection_name='amazon_books_collection_sampled',
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [15]:
print("All points in the collection:", all_points)
print(len(all_points[0]))


All points in the collection: ([Record(id=0, payload={'description': 'Truth is the Soul of the Sun - A Biographical Novel of Hatshepsut-MaatkareREVISED SECOND EDITIONThis is the story of Hatshepsut, the ancient Egyptian princess and queen who was crowned king, and for more than twenty years ruled on the Horus Throne of the Living as the Female Falcon. The name Hatshepsut chose for herself can be read as "Maat is the Ka of Re" which translated means "The True and Beautiful Manifestation of the Sun’s Divine Life-force." A more poetic but still accurate rendition is "Truth is the Soul of the Sun."Before becoming Pharaoh, Hatshepsut served as God’s Wife of Amun, an important economic and spiritual office created by her grandfather that holds the key to her mysterious and unprecedented power. Maatkare was not only a charismatic political leader, she was a mystic who inspired the wholehearted devotion of brilliant men. One of them was Senmut, a commoner she elevated to unparalleled heights o

In [17]:
all_context = [{'id':data.payload['parent_asin'], 'text':data.payload['description']} for data in all_points[0]]

In [18]:
all_context

[{'id': 'B002HWRUFW',
  'text': 'Truth is the Soul of the Sun - A Biographical Novel of Hatshepsut-MaatkareREVISED SECOND EDITIONThis is the story of Hatshepsut, the ancient Egyptian princess and queen who was crowned king, and for more than twenty years ruled on the Horus Throne of the Living as the Female Falcon. The name Hatshepsut chose for herself can be read as "Maat is the Ka of Re" which translated means "The True and Beautiful Manifestation of the Sun’s Divine Life-force." A more poetic but still accurate rendition is "Truth is the Soul of the Sun."Before becoming Pharaoh, Hatshepsut served as God’s Wife of Amun, an important economic and spiritual office created by her grandfather that holds the key to her mysterious and unprecedented power. Maatkare was not only a charismatic political leader, she was a mystic who inspired the wholehearted devotion of brilliant men. One of them was Senmut, a commoner she elevated to unparalleled heights of authority. But her most influential

#### Render a prompt to generate synthetic Eval reference dataset


In [23]:
output_schema = {
    'type':'array',
    'items':{
        'type':'object',
        'properties':{
            'question':{
                'type':'string',
                'description':'Suggested question.',
            },
            'chunk_ids':{
                'type':'array',
                'items':{
                    'type':'string',
                    'description':'ID of the chunk that could be used to answer the question.',
                },   
            },
            'answer_example':{
                'type':'string',
                'description':'Suggested answer grounded in the context',
            },
            'reasoning':{
                'type':'string',
                'description':'Reasoning why the question could be answered with the chunks.',
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answer could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also provide the reason why you chose the chunks to answer the questions.
Construct 10 that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
<OUTPUT JSON SCHEMA>
I need to be able to parse the json output
"""

USER_PROMPT = f"""
Here is the list of chunks of text that you can use to answer the questions, each list element is a dictionary with id and text:
{all_context}
"""

In [25]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answer could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also provide the reason why you chose the chunks to answer the questions.
Construct 10 that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{
  "type": "array",
  "items": {
    "type": "objec

In [26]:
print(USER_PROMPT)


Here is the list of chunks of text that you can use to answer the questions, each list element is a dictionary with id and text:
[{'id': 'B002HWRUFW', 'text': 'Truth is the Soul of the Sun - A Biographical Novel of Hatshepsut-MaatkareREVISED SECOND EDITIONThis is the story of Hatshepsut, the ancient Egyptian princess and queen who was crowned king, and for more than twenty years ruled on the Horus Throne of the Living as the Female Falcon. The name Hatshepsut chose for herself can be read as "Maat is the Ka of Re" which translated means "The True and Beautiful Manifestation of the Sun’s Divine Life-force." A more poetic but still accurate rendition is "Truth is the Soul of the Sun."Before becoming Pharaoh, Hatshepsut served as God’s Wife of Amun, an important economic and spiritual office created by her grandfather that holds the key to her mysterious and unprecedented power. Maatkare was not only a charismatic political leader, she was a mystic who inspired the wholehearted devotion 

In [31]:
response = openai.chat.completions.create(
    model = 'gpt-5-mini',
    messages=[
        {'role': 'system', 'content':SYSTEM_PROMPT},
        {'role': 'user', 'content':USER_PROMPT}
    ],
    reasoning_effort='minimal' 
)

print(response.choices[0].message.content)

[
  {
    "question": "Do you have any books about ancient Egypt or pharaohs, particularly Hatshepsut?",
    "chunk_ids": [
      "B002HWRUFW",
      "B08694WNQJ"
    ],
    "answer_example": "Yes. We have 'Truth is the Soul of the Sun - A Biographical Novel of Hatshepsut-Maatkare' (ID B002HWRUFW), a biographical novel about Hatshepsut, and 'Scarab in the Storm (The Lost Pharaoh Chronicles Book 3)' (ID B08694WNQJ), which focuses on Tutankhamun and Queen Ankhesenamun in a fictionalized political setting.",
    "reasoning": "B002HWRUFW explicitly covers Hatshepsut and her biography. B08694WNQJ is a related Egypt/pharaoh novel (Tut/Ankhesenamun), so both chunks are relevant for a user asking about books on ancient Egyptian rulers."
  },
  {
    "question": "Which novels in stock are set during World War II and involve villages or French vineyards?",
    "chunk_ids": [
      "B07J2DY4KW",
      "1982178779"
    ],
    "answer_example": "We have 'The Forgotten Village' (ID B07J2DY4KW), a Wo

In [32]:
json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [33]:
json_output

[{'question': 'Do you have any books about ancient Egypt or pharaohs, particularly Hatshepsut?',
  'chunk_ids': ['B002HWRUFW', 'B08694WNQJ'],
  'answer_example': "Yes. We have 'Truth is the Soul of the Sun - A Biographical Novel of Hatshepsut-Maatkare' (ID B002HWRUFW), a biographical novel about Hatshepsut, and 'Scarab in the Storm (The Lost Pharaoh Chronicles Book 3)' (ID B08694WNQJ), which focuses on Tutankhamun and Queen Ankhesenamun in a fictionalized political setting.",
  'reasoning': 'B002HWRUFW explicitly covers Hatshepsut and her biography. B08694WNQJ is a related Egypt/pharaoh novel (Tut/Ankhesenamun), so both chunks are relevant for a user asking about books on ancient Egyptian rulers.'},
 {'question': 'Which novels in stock are set during World War II and involve villages or French vineyards?',
  'chunk_ids': ['B07J2DY4KW', '1982178779'],
  'answer_example': "We have 'The Forgotten Village' (ID B07J2DY4KW), a World War II historical fiction about Tyneham, and 'The Winemaker

In [34]:
len(json_output)

37

In [38]:
points = qdrant_client.scroll(
    collection_name='amazon_books_collection_sampled',
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key='parent_asin',
                match = MatchValue(value='1455571431')
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [39]:
points[0].payload

{'description': 'We Were Strangers OnceFor fans of The Nightingale and Brooklyn comes an exquisite and unforgettable novel about friendship, love, and redemption in a circle of immigrants who flee Europe for 1930s-era New York City. "Carter\'s warm and beautiful prose brings us love, tragedy, mystery and hope in a moving celebration of America and the people who have come to it." -- Amy Bloom, New York Times bestselling author of Lucky Us and Away On the eve of World War II Egon Schneider--a gallant and successful Jewish doctor, son of two world-famous naturalists--escapes Germany to an uncertain future across the sea. Settling into the unfamiliar rhythms of upper Manhattan, he finds solace among a tight-knit group of fellow immigrants, tenacious men and women drawn together as much by their differences as by their memories of the world they left behind. They each suffer degradations and triumphs large and small: Egon\'s terminally acerbic lifelong friend, bestselling author Meyer Leav

In [42]:
def get_description(parent_asin: str)-> str:
    points = qdrant_client.scroll(
        collection_name='amazon_books_collection_sampled',
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key='parent_asin',
                    match = MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload['description']

In [43]:
get_description('B01DWTHKFI')

'A Kingdom\'s Cost: A Historical Novel of Scotland (The Black Douglas Trilogy Book 1)• " I was unable to put A Kingdom\'s Cost down as I read it, and I miss the adventure even now... " - Online Book Club Editorial Review Eighteen-year-old James Douglas can only watch, helpless, as the Scottish freedom fighter, William Wallace, is hanged, drawn, and quartered. Even under the heel of a brutal English conqueror, James\'s blood-drenched homeland may still have one hope for freedom, the rightful King of the Scots, Robert the Bruce. James swears fealty to the man he believes can lead the fight against English tyranny. The Bruce is soon a fugitive, king in name only. Scotland is occupied, the Scottish resistance crushed. The woman James loves is captured and imprisoned. Yet James believes their cause is not lost. With driving determination, he blazes a path in blood and violence, in cunning and ruthlessness as he wages a guerrilla war to restore Scotland\'s freedom. James knows he risks shari

#### Create Eval dataset in Langsmith

In [44]:
client = Client(api_key=os.environ['LANGSMITH_API_KEY'])

In [45]:
dataset_name = 'rag-evaluation-dataset'
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description='Dataset for evaluating RAG pipeline'
)

In [47]:
for item in json_output:
    client.create_example(
        dataset_id=dataset.id,
        inputs ={'question':item['question']},
        reference_outputs={
            'ground_truth': item['answer_example'],
            'reference_context_ids': item['chunk_ids'],
            'reference_descriptions':[get_description(id) for id in item['chunk_ids']]
        }
    )